In [1]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import muon as mu
from muon import atac as ac
from pyjaspar import jaspardb
import pychromvar as pc


In [9]:
loc_mm10 = '/data2st1/junyi/ref/GRCm38.p6.genome.fa'

In [2]:
adata=sc.read_h5ad('/data1st2/junyi/output/atac1112/subset/region_nt/PFC_PFC_GABA.h5ad')

In [3]:
adata.obs

,sample,doublet_probability,doublet_score,leiden,leiden_default,leiden_res_0.1,leiden_res_0.2,leiden_res_0.3,leiden_res_0.4,leiden_res_0.5,...,celltype.L1_ct,Sample_name,Condition,Region,celltype.L2.raw,region_nt,celltype.L3,celltype.L4,celltype.L2.refined,expriment
MC50B_PFC:AAACTGCCACGGTTAT-1,MC50B_PFC,0.121567,0.035714,4,1,0,0,1,1,1,...,GABA,MC50B_PFC,MC,PFC,PFC Pvalb GABA,PFC_GABA,PFC_Pvalb_GABA-1,PFC_Pvalb_GABA-1-0,PFC Pvalb GABA,MC
MC50B_PFC:AAACTGCGTCGATTAC-1,MC50B_PFC,0.138635,0.024831,4,1,0,0,1,1,1,...,GABA,MC50B_PFC,MC,PFC,PFC Pvalb GABA,PFC_GABA,PFC_Pvalb_GABA-0,PFC_Pvalb_GABA-0-0,PFC Pvalb GABA,MC
MC50B_PFC:AAAGGATGTGTCGGTC-1,MC50B_PFC,0.141703,0.023256,4,1,0,0,1,1,1,...,GABA,MC50B_PFC,MC,PFC,PFC Sncg GABA,PFC_GABA,PFC_Sncg_GABA-0,PFC_Sncg_GABA-0-0,PFC Sncg GABA,MC
MC50B_PFC:AAAGGATTCCTCCATG-1,MC50B_PFC,0.138635,0.024831,4,1,0,0,1,1,1,...,GABA,MC50B_PFC,MC,PFC,PFC Pvalb GABA,PFC_GABA,PFC_Pvalb_GABA-0,PFC_Pvalb_GABA-0-0,PFC Pvalb GABA,MC
MC50B_PFC:AAAGGGCAGATGCGAC-1,MC50B_PFC,0.157649,0.016194,4,1,0,0,1,1,1,...,GABA,MC50B_PFC,MC,PFC,PFC Sst GABA,PFC_GABA,PFC_Sst_GABA-1,PFC_Sst_GABA-1-0,PFC Sst GABA,MC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
MW26A_PFC:TTTGTGTAGGTTGTGG-1,MW26A_PFC,0.388409,0.010309,4,1,0,0,1,1,1,...,GABA,MW26A_PFC,MW,PFC,PFC Vip GABA,PFC_GABA,PFC_Vip_GABA-1,PFC_Vip_GABA-1-0,PFC Vip GABA,MW
MW26A_PFC:TTTGTGTGTCCTTATT-1,MW26A_PFC,0.354129,0.014599,4,1,0,0,1,1,1,...,GABA,MW26A_PFC,MW,PFC,PFC Sst GABA,PFC_GABA,PFC_Sst_GABA-0,PFC_Sst_GABA-0-0,PFC Sst GABA,MW
MW26A_PFC:TTTGTGTGTGAAACAT-1,MW26A_PFC,0.283147,0.026094,4,1,0,0,1,1,1,...,GABA,MW26A_PFC,MW,PFC,PFC Vip GABA,PFC_GABA,PFC_Vip_GABA-1,PFC_Vip_GABA-1-0,PFC Vip GABA,MW
MW26A_PFC:TTTGTGTGTGATGTGG-1,MW26A_PFC,0.404006,0.008521,4,1,0,0,1,1,1,...,GABA,MW26A_PFC,MW,PFC,PFC Sst GABA,PFC_GABA,PFC_Sst_GABA-0,PFC_Sst_GABA-0-0,PFC Sst GABA,MW


In [18]:
adata.var['chr'] = adata.var.index

In [19]:
adata.var.index = adata.var['chr'].str.replace(':', '-')

In [43]:
adata.X = adata.layers['count'] # use norm matrix for chromvar

In [44]:
sc.pp.calculate_qc_metrics(adata, percent_top=None, log1p=False, inplace=True)


In [49]:
mu.pp.filter_var(adata, 'n_cells_by_counts', lambda x: x >= 50)
mu.pp.filter_obs(adata, 'n_genes_by_counts', lambda x: (x >= 2000) )
mu.pp.filter_obs(adata, 'total_counts', lambda x: (x >= 4000))


In [50]:
adata

AnnData object with n_obs × n_vars = 6326 × 642438
    obs: 'sample', 'doublet_probability', 'doublet_score', 'leiden', 'leiden_default', 'leiden_res_0.1', 'leiden_res_0.2', 'leiden_res_0.3', 'leiden_res_0.4', 'leiden_res_0.5', 'leiden_res_0.6', 'leiden_res_0.7', 'leiden_res_0.8', 'leiden_res_0.9', 'leiden_res_1.0', 'leiden_res_1.1', 'leiden_res_1.2', 'leiden_res_1.3', 'leiden_res_1.4', 'leiden_res_1.5', 'leiden_res_1.6', 'leiden_res_1.7', 'leiden_res_1.8', 'leiden_res_1.9', 'celltype.L2.Condition', 'celltype.L1', 'celltype.L2', 'Neurotransmitter_celltype', 'celltype.L1_ct', 'Sample_name', 'Condition', 'Region', 'celltype.L2.raw', 'region_nt', 'celltype.L3', 'celltype.L4', 'celltype.L2.refined', 'expriment', 'n_genes_by_counts', 'total_counts'
    var: 'chr', 'gc_bias', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'
    uns: 'log1p', 'peak_seq'
    obsm: 'X_spectral', 'X_umap'
    layers: 'count'

In [52]:
pc.add_peak_seq(adata, genome_file=loc_mm10)


100%|██████████| 642438/642438 [00:02<00:00, 215873.24it/s]


In [53]:
pc.add_gc_bias(adata)

100%|██████████| 642438/642438 [00:03<00:00, 190363.66it/s]
/home/junyichen/anaconda3/envs/scenicplus/lib/python3.11/site-packages/pychromvar/preprocessing.py:134: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  adata.var['gc_bias'] = bias


In [34]:
jdb_obj = jaspardb(release='JASPAR2026')
motifs = jdb_obj.fetch_motifs(
    collection = 'CORE',
    tax_group = ['vertebrates'])


In [54]:
pc.get_bg_peaks(adata)


/home/junyichen/anaconda3/envs/scenicplus/lib/python3.11/site-packages/scipy/sparse/_index.py:145: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil_matrix is more efficient.
  self._set_arrayXarray(i, j, x)


In [59]:
adata.X = adata.X.astype(np.float32)

In [61]:
pc.match_motif(adata, motifs=motifs)

100%|██████████| 642438/642438 [07:28<00:00, 1433.61it/s]


In [71]:
len(adata.uns['motif_name'])

1019

In [75]:
adata.varm['motif_match'].shape

(642438, 1019)

In [76]:
df_motif_match = pd.DataFrame(adata.varm['motif_match'], index=adata.var_names, columns=adata.uns['motif_name'])

In [78]:
dev = pc.compute_deviations(adata)


2026-06-08 15:15:29 INFO     computing expectation reads per cell and peak...
2026-06-08 15:20:10 INFO     computing observed motif deviations...
2026-06-08 15:20:38 INFO     computing background deviations...
/home/junyichen/anaconda3/envs/scenicplus/lib/python3.11/site-packages/anndata/_core/anndata.py:522: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


In [85]:
dev.var

""
MA0004.1.Arnt
MA0069.1.PAX6
MA0071.1.RORA
MA0074.1.RXRA::VDR
MA0101.1.REL
...
MA0505.3.Nr5a2
MA0506.3.Nrf1
MA1618.2.Ptf1A
MA0002.3.Runx1
